In [2]:
import mlflow
from mlflow.tracking import MlflowClient
import tempfile
import time
import json

from configuration import MLFLOW_TRACKING_URI, MODEL_NAME

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

client = MlflowClient()
model_version = None
model_uri = None
for i in range(10):
    try:
        print(f"Trying search model in MLflow... attempt {i+1}")
        models = client.search_registered_models(filter_string=f"name='{MODEL_NAME}'")
        
        if not models:
            print(f"Model '{MODEL_NAME}' not found in MLflow")
            time.sleep(2)
            continue
        
        registered_model = models[0]
        aliases = registered_model.aliases
        
        champion_version = aliases.get("champion")
        
        if champion_version is None:
            print(f"No version with 'champion' alias found for model '{MODEL_NAME}'")
            continue

        print(f"Found champion version: {champion_version}")
        
        model_version = client.get_model_version(MODEL_NAME, champion_version)
        model_uri = model_version.source
        break
    except Exception as e:
        print("Retrying MLflow...", e)
        time.sleep(2)
        
else:
    print(f"Failed to find model '{MODEL_NAME}' with 'champion' alias after multiple attempts")
    raise RuntimeError("Failed to get model from MLflow")



for i in range(10):
    try:
        print(f"Trying to load model from MLflow attempt {i+1}, uri: {model_uri}")
        model = mlflow.pyfunc.load_model(model_uri)
        break
    except Exception as e:
        print("Retrying MLflow...", e)
        time.sleep(2)

if model is None:
    print("Failed to load model from MLflow after multiple attempts")
    raise RuntimeError("Failed to load model from MLflow")

# load schema
run_id = model_version.run_id

with tempfile.TemporaryDirectory() as tmpdir:
    print(f"Downloading schema.json from MLflow run_id: {run_id} to temporary directory: {tmpdir}")
    schema_path = client.download_artifacts(run_id, "schema.json", dst_path=tmpdir)
    print(f"Schema downloaded to: {schema_path}")
    with open(schema_path) as f:
        schema = json.load(f)
        print(f"Schema loaded: {schema}")

Trying search model in MLflow... attempt 1
Found champion version: 6
Trying to load model from MLflow attempt 1, uri: models:/m-c4ad7790e60744009d614e7543b1f00e


Schema downloaded to: C:\Users\yeech\AppData\Local\Temp\tmptjc2hali\schema.json
Schema loaded: {'features': {'numerical': [{'name': 'tenure', 'type': 'int64'}, {'name': 'MonthlyCharges', 'type': 'float64'}, {'name': 'TotalCharges', 'type': 'float64'}], 'categorical': [{'name': 'gender', 'values': ['Female', 'Male']}, {'name': 'SeniorCitizen', 'values': [0, 1]}, {'name': 'Partner', 'values': ['Yes', 'No']}, {'name': 'Dependents', 'values': ['No', 'Yes']}, {'name': 'PhoneService', 'values': ['No', 'Yes']}, {'name': 'MultipleLines', 'values': ['No phone service', 'No', 'Yes']}, {'name': 'InternetService', 'values': ['DSL', 'Fiber optic', 'No']}, {'name': 'OnlineSecurity', 'values': ['No', 'Yes', 'No internet service']}, {'name': 'OnlineBackup', 'values': ['Yes', 'No', 'No internet service']}, {'name': 'DeviceProtection', 'values': ['No', 'Yes', 'No internet service']}, {'name': 'TechSupport', 'values': ['No', 'Yes', 'No internet service']}, {'name': 'StreamingTV', 'values': ['No', 'Yes', 

In [10]:
from app.request_model import create_request_model
from polyfactory.factories.pydantic_factory import ModelFactory
from httpx import Client

request_model = create_request_model(schema)

class InputCreateFactory(ModelFactory[request_model]):
    __model__ = request_model

api_url = "http://56.69.215.81:8000"
test_count = 99

http_client = Client(base_url=api_url)

for i in range(test_count):
    input_data = InputCreateFactory.build()
    payload = input_data.model_dump(mode="json")
        
    resp = http_client.post("/predict", json=payload)
    print(resp.status_code, resp.json()) 

2026-09-11 22:44:40,738 - INFO - Creating request model from schema...
2026-09-11 22:44:40,741 - INFO - Request model created successfully


200 {'prediction': 'Yes', 'probability': 1.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'Yes', 'probability': 1.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 0.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 0.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 0.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 1.1562976225393598e-253, 'threshold': 0.2799999999999999}
200 {'prediction': 'Yes', 'probability': 1.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 0.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'Yes', 'probability': 1.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'Yes', 'probability': 1.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 2.761213149565261e-27, 'threshold': 0.2799999999999999}
200 {'prediction': 'No', 'probability': 0.0, 'threshold': 0.2799999999999999}
200 {'prediction': 'N